In [1]:
import os
from dotenv import load_dotenv

load_dotenv()
deepseek_api= os.getenv("deepseek_api_key")

In [3]:
import json
from openai import OpenAI

# 初始化客户端 (DeepSeek API 兼容 OpenAI 格式)
client = OpenAI(
    api_key=deepseek_api, 
    base_url="https://api.deepseek.com"
)

def generate_prompts(count=100):
    prompts_list = []
    
    # 分批次生成，防止 API 超时或单次生成过多导致质量下降
    # 每次请求生成 10 个，循环 10 次
    batch_size = 10
    loop=10

    print(f"开始生成 {count} 个测试 Prompt...")

    system_content = (
        "你是一个辅助实验的助手。你需要提供多样化的写作指令（Prompts）。"
        "这些指令应涵盖：创意写作、科普解释、代码编写建议、哲学讨论、新闻摘要等。"
        "要求：每个指令必须能诱导模型写出 200 字以上的长文本。"
        "输出格式：仅输出一个 JSON 数组，包含字符串。"
    )

    for i in range(10):
        user_content = f"请提供 {batch_size} 个互不相同的、高质量的 Prompt。不要重复，不要包含序号。"
        
        try:
            response = client.chat.completions.create(
                model="deepseek-chat",
                messages=[
                    {"role": "system", "content": system_content},
                    {"role": "user", "content": user_content},
                ],
                response_format={'type': 'json_object'} # 强制返回 JSON
            )
            
            # 这里的解析逻辑取决于 DeepSeek 返回的具体结构
            # 通常返回的是 {"prompts": ["...", "..."]}
            batch_data = json.loads(response.choices[0].message.content)
            
            # 兼容处理：提取列表
            if isinstance(batch_data, dict):
                for key in batch_data:
                    if isinstance(batch_data[key], list):
                        prompts_list.extend(batch_data[key])
                        break
            elif isinstance(batch_data, list):
                prompts_list.extend(batch_data)

            print(f"已完成: {len(prompts_list)}/{count}")
            
        except Exception as e:
            print(f"批次 {i+1} 生成失败: {e}")

    # 截取前 100 个并保存
    final_prompts = prompts_list[:count]
    
    with open("prompts.json", "w", encoding="utf-8") as f:
        json.dump({"test_prompts": final_prompts}, f, ensure_ascii=False, indent=4)
    
    print(f"成功保存至 prompts.json，共 {len(final_prompts)} 个条目。")

if __name__ == "__main__":
    generate_prompts(100)

开始生成 100 个测试 Prompt...
已完成: 10/100
已完成: 20/100
已完成: 30/100
已完成: 40/100
已完成: 50/100
已完成: 60/100
已完成: 70/100
已完成: 80/100
已完成: 90/100
已完成: 100/100
成功保存至 prompts.json，共 100 个条目。


In [4]:
from datasets import load_dataset

# 加载英文维基百科（或中英文混搭）
ds = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")

# 过滤掉空行和太短的段落
human_texts = [text.strip() for text in ds["text"] if len(text.split()) > 150][:100]

with open("human_texts.json", "w", encoding="utf-8") as f:
    json.dump({"human_texts": human_texts}, f, ensure_ascii=False, indent=4)

c:\Users\zswissac\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\zswissac\AppData\Local\Programs\Python\Python314\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\zswissac\.cache\huggingface\hub\datasets--wikitext. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an admin